# A parameter sweep and its heat map

Run a grid of rule settings and look at a region, not a single best cell: each row's deflated Sharpe ratio counts every combination as a trial.

In [ ]:
# A temporary workspace and a synthetic daily recording, so the notebook runs anywhere.
import csv, datetime, json, random
from pathlib import Path
from tempfile import mkdtemp

from sim_platform import Platform

folder = Path(mkdtemp(prefix='market-lab-notebook-'))
rng = random.Random(7)
price = 100.0
recording = folder / 'acme.csv'
with recording.open('w', newline='') as stream:
    out = csv.writer(stream)
    out.writerow(['Date', 'Open', 'High', 'Low', 'Close', 'Volume'])
    for day in range(40):
        date = datetime.date(2021, 3, 1) + datetime.timedelta(days=day)
        close = price * (1 + rng.gauss(0.0005, 0.012))
        out.writerow([date.isoformat(), round(price, 2), round(max(price, close) + 0.5, 2),
                      round(min(price, close) - 0.5, 2), round(close, 2), 1_000_000])
        price = close
app = Platform(folder / 'workspace')
profile = app.create_profile('Notebook')
print(recording)

In [ ]:
from sim_platform.research.sweeps import heatmap, render_heatmap_svg

DAY = 86_400 * 10**9
result = app.run_sweep([str(recording)], dict(name='rule', policy='rule'),
                       dict(quantity=[100, 300], hold_ns=[2 * DAY, 5 * DAY, 10 * DAY]),
                       profile=profile, symbol='ACME', cash=1_000_000_000, resamples=500)
for row in result['rows']:
    print(row['name'], round(row['mean'], 4))
svg = render_heatmap_svg(heatmap(result['rows'], 'hold_ns', 'quantity'), title='Total return')
(folder / 'heatmap.svg').write_text(svg)
try:
    from IPython.display import SVG, display
    display(SVG(svg))
except ImportError:
    print('heat map written to', folder / 'heatmap.svg')